# Загрузка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
import os

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.figsize'] = (14, 6)

print("Работаем на уровне сетевых потоков (flows)")

print(f"\nЗагрузка данных...")
df_train = pd.read_csv('../data/processed/train_raw.csv')
df_test = pd.read_csv('../data/processed/test_raw.csv')

print(f"Train: {len(df_train):,} записей")
print(f"Test: {len(df_test):,} записей")

Работаем на уровне сетевых потоков (flows)

Загрузка данных...
Train: 175,341 записей
Test: 82,332 записей


# Предобработка

In [2]:
print(f"Пропуски в train: {df_train.isnull().sum().sum()}")
print(f"Пропуски в test: {df_test.isnull().sum().sum()}")

df_train = df_train.fillna(0)
df_test = df_test.fillna(0)

df_train['target_anomaly'] = df_train['label'].astype(int)
df_test['target_anomaly'] = df_test['label'].astype(int)

if 'attack_cat' in df_train.columns:
    le = LabelEncoder()
    all_cats = pd.concat([df_train['attack_cat'], df_test['attack_cat']]).fillna('Normal')
    le.fit(all_cats)
    
    df_train['target_attack_type'] = le.transform(df_train['attack_cat'].fillna('Normal'))
    df_test['target_attack_type'] = le.transform(df_test['attack_cat'].fillna('Normal'))
    
    print(f"\nТипы атак ({len(le.classes_)}):")
    for i, cat in enumerate(le.classes_):
        print(f"  {i}: {cat}")

print(f"\nTrain - аномалий: {df_train['target_anomaly'].sum():,} ({df_train['target_anomaly'].mean()*100:.1f}%)")
print(f"Test - аномалий: {df_test['target_anomaly'].sum():,} ({df_test['target_anomaly'].mean()*100:.1f}%)")

Пропуски в train: 0
Пропуски в test: 0

Типы атак (10):
  0: Analysis
  1: Backdoor
  2: DoS
  3: Exploits
  4: Fuzzers
  5: Generic
  6: Normal
  7: Reconnaissance
  8: Shellcode
  9: Worms

Train - аномалий: 119,341 (68.1%)
Test - аномалий: 45,332 (55.1%)


# Выбор признаков

In [3]:
# Исключаем служебные колонки
exclude_cols = ['id', 'label', 'attack_cat', 'target_anomaly', 'target_attack_type']
feature_cols = [col for col in df_train.columns if col not in exclude_cols]

print(f"Используем {len(feature_cols)} признаков из UNSW-NB15")
print(f"Первые 10 признаков: {feature_cols[:10]}")

X_train = df_train[feature_cols]
y_train_anomaly = df_train['target_anomaly']
y_train_attack_type = df_train['target_attack_type'] if 'target_attack_type' in df_train.columns else None

X_test = df_test[feature_cols]
y_test_anomaly = df_test['target_anomaly']
y_test_attack_type = df_test['target_attack_type'] if 'target_attack_type' in df_test.columns else None

print(f"\nX_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

Используем 42 признаков из UNSW-NB15
Первые 10 признаков: ['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl']

X_train: (175341, 42)
X_test: (82332, 42)


# Baseline для детекции аномалий

In [4]:
y_pred_always_normal = np.zeros(len(y_test_anomaly))

total_bytes_train = X_train['sbytes'] + X_train['dbytes']
threshold_bytes = total_bytes_train.mean()
total_bytes_test = X_test['sbytes'] + X_test['dbytes']
y_pred_threshold = (total_bytes_test > threshold_bytes).astype(int)

threshold_dur = X_train['dur'].quantile(0.95)
y_pred_dur = (X_test['dur'] > threshold_dur).astype(int)

def get_metrics(y_true, y_pred):
    return {
        'Accuracy': f"{accuracy_score(y_true, y_pred):.3f}",
        'Precision': f"{precision_score(y_true, y_pred, zero_division=0):.3f}",
        'Recall': f"{recall_score(y_true, y_pred, zero_division=0):.3f}",
        'F1': f"{f1_score(y_true, y_pred, zero_division=0):.3f}"
    }

baselines = {
    'Always Normal': y_pred_always_normal,
    f'Threshold (bytes > {threshold_bytes:.0f})': y_pred_threshold,
    f'Threshold (dur > {threshold_dur:.2f})': y_pred_dur
}

results = []
for name, y_pred in baselines.items():
    metrics = get_metrics(y_test_anomaly, y_pred)
    metrics['Model'] = name
    results.append(metrics)

results_df = pd.DataFrame(results).set_index('Model')
display(results_df)

,Accuracy,Precision,Recall,F1
Model,,,,
Always Normal,0.449,0.000,0.000,0.000
Threshold (bytes > 23774),0.414,0.321,0.058,0.099
Threshold (dur > 3.08),0.453,0.542,0.037,0.069


# Сохранение данных

In [5]:
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

y_train_anomaly.to_csv('../data/processed/y_train_anomaly.csv', index=False)
y_test_anomaly.to_csv('../data/processed/y_test_anomaly.csv', index=False)

if y_train_attack_type is not None:
    y_train_attack_type.to_csv('../data/processed/y_train_attack_type.csv', index=False)
    y_test_attack_type.to_csv('../data/processed/y_test_attack_type.csv', index=False)

with open('../data/processed/feature_cols.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print(f"Сохранено:")
print(f"  - X_train.csv ({X_train.shape})")
print(f"  - X_test.csv ({X_test.shape})")
print(f"  - y_train_anomaly.csv ({len(y_train_anomaly):,})")
print(f"  - y_test_anomaly.csv ({len(y_test_anomaly):,})")
print(f"  - feature_cols.txt ({len(feature_cols)} признаков)")

Сохранено:
  - X_train.csv ((175341, 42))
  - X_test.csv ((82332, 42))
  - y_train_anomaly.csv (175,341)
  - y_test_anomaly.csv (82,332)
  - feature_cols.txt (42 признаков)


# Сохранение данных

In [6]:
os.makedirs('../data/processed', exist_ok=True)

df_train.to_csv('../data/processed/train_final.csv', index=False)
df_test.to_csv('../data/processed/test_final.csv', index=False)

# Сохраняем список признаков
with open('../data/processed/feature_cols.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print(f"Сохранено:")
print(f"  - train_final.csv ({len(df_train)} интервалов)")
print(f"  - test_final.csv ({len(df_test)} интервалов)")
print(f"  - feature_cols.txt ({len(feature_cols)} признаков)")

Сохранено:
  - train_final.csv (175341 интервалов)
  - test_final.csv (82332 интервалов)
  - feature_cols.txt (42 признаков)
